# Recommendation system models

### Libraries Import

In [1]:
import sqlite3
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

### Dataset Import

##### Users Dataset

In [2]:
conn = sqlite3.connect("../fastapi_recommender/amazon_electronics.db")
users_df = pd.read_sql_query("SELECT * FROM users", conn)
conn.close()

users_df.head()

,user_id,user_name,user_pass,Country,Age,City,Marital_Status
0,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...",pass123,India,60,Delhi,Single
1,"AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...",pass123,Brazil,60,Brasilia,Widowed
2,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...",pass123,Australia,28,Brisbane,Married
3,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...",pass123,Portugal,58,Lisbon,Married
4,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...",pass123,USA,23,Chicago,Single


##### Products Dataset

In [3]:
conn = sqlite3.connect("../fastapi_recommender/amazon_electronics.db")
products_df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

products_df.head()


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating_count,about_product,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1.099,0.64,24269,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.000,0.43,43994,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1.899,0.90,7928,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.000,0.53,94363,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.000,0.61,16905,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


##### Ratings Dataset

In [4]:
conn = sqlite3.connect("../fastapi_recommender/amazon_electronics.db")

ratings_df = pd.read_sql_query(
    "SELECT user_id, product_id, rating FROM ratings WHERE rating IS NOT NULL",
    conn
)

conn.close()

ratings_df.head()

,user_id,product_id,rating
0,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...",B07JW9H4J1,4.2
1,"AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...",B098NS6PVG,4.0
2,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...",B096MSW6CT,3.9
3,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...",B08HDJ86NZ,4.2
4,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...",B08CF3B7N1,4.2


In [5]:
ratings_df.info()
ratings_df["rating"].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5962 entries, 0 to 5961
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   user_id     5962 non-null   object 
 1   product_id  5962 non-null   object 
 2   rating      5962 non-null   float64
dtypes: float64(1), object(2)
memory usage: 139.9+ KB


count    5962.000000
mean        4.047920
std         0.646874
min         1.800000
25%         3.600000
50%         4.100000
75%         4.500000
max         5.000000
Name: rating, dtype: float64

### Non personalized recommender

In [6]:
merged_df = pd.merge(
    products_df,
    ratings_df,
    on="product_id"
)

top_products = (
    merged_df
    .groupby(
        ['product_id', 'product_name', 'discounted_price']
    )['rating']
    .mean()
    .reset_index(name='avg_rating')
    .sort_values(by='avg_rating', ascending=False)
    .head(5)
)

top_products['avg_rating'] = top_products['avg_rating']

top_products

,product_id,product_name,discounted_price,avg_rating
1348,B0BQRJ3C47,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",249.000,5.0
1120,B09ZHCJDP1,Amazon Basics Wireless Mouse | 2.4 GHz Connect...,499.000,5.0
1341,B0BP7XLX48,Syncwire LTG to USB Cable for Fast Charging Co...,399.000,5.0
1349,B0BR4F878Q,Swiffer Instant Electric Water Heater Faucet T...,1.439,4.8
1347,B0BQ3K23Y1,"Oratech Coffee Frother electric, milk frother ...",279.000,4.8


### Content based filtering

In [7]:
# ── 1. Category encoding ────────────────────────────
products_df["category_split"] = (
    products_df["category"]
    .str.split(r"[|&]")
    .apply(lambda x: list(set(x)))
)
mlb = MultiLabelBinarizer()
category_matrix = csr_matrix(mlb.fit_transform(products_df["category_split"]))

# ── 2. Price scaling ─────────────────────────────────
scaler = MinMaxScaler()
price_scaled = csr_matrix(
    scaler.fit_transform(products_df[["discounted_price"]])
)

# ── 3. TF-IDF on product_name ─────────────────────────────────────────────────
tfidf_name = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams catch "fast charging", "Type-C", etc.
    max_features=500
)
name_matrix = tfidf_name.fit_transform(
    products_df["product_name"].fillna("")
)

# ── 4. TF-IDF on about_product ────────────────────────────────────────────────
tfidf_about = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=1000     # richer descriptions → more features
)
about_matrix = tfidf_about.fit_transform(
    products_df["about_product"].fillna("")
)

In [8]:
# ── 5. Weighted feature matrix ────────────────────────────────────────────────
W_CATEGORY = 2.0
W_PRICE    = 0.5
W_NAME     = 1.5
W_ABOUT    = 1.0

feature_matrix = hstack([
    category_matrix * W_CATEGORY,
    price_scaled    * W_PRICE,
    name_matrix     * W_NAME,
    about_matrix    * W_ABOUT,
])

# ── 6. Similarity matrix ──────────────────────────────────────────────────────
similarity_matrix = cosine_similarity(feature_matrix)

In [9]:
# função para obter os produtos mais semelhantes a um produto específico
def get_nearest_products(product_id, similarity_matrix, products_df, k):
    product_index_map = pd.Series(
        products_df.index,
        index=products_df["product_id"]
    ).to_dict()

    if product_id not in product_index_map:
        return {}

    index = product_index_map[product_id]

    similarity_scores = list(enumerate(similarity_matrix[index]))
                                                                                

    similarity_scores = sorted(      
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_products = similarity_scores[1:k+1]

    k_nearest = {
        products_df.iloc[i]["product_id"]: score
        for i, score in top_products
    }

    return k_nearest

In [10]:
get_nearest_products(
    "B07YTNKVJQ",
    similarity_matrix,
    products_df,
    k=5
)

{'B09X79PP8F': np.float64(0.9435025251581326),
 'B0B4HKH19N': np.float64(0.9176682182685089),
 'B08DDRGWTJ': np.float64(0.916310464659329),
 'B083342NKJ': np.float64(0.9062356054012883),
 'B01GGKZ0V6': np.float64(0.9041898073467971)}

In [11]:
rated_produnts = {'B0789LZTCJ': 4.2, 'B094JNXNPV': 3.5}

def get_recommendations(rated_products, n):     

    candidates = {}

    for p, r in rated_products.items():

        k_nearest = get_nearest_products(p,similarity_matrix,products_df,n*2)

        for product, cos_sim in k_nearest.items():
            if product in rated_products:
                continue
            if product in candidates:
                candidates[product] += float(cos_sim)  
            else:                                      
                candidates[product] = float(cos_sim) * r
    
    recommendations = [
        product_id 
        for product_id, _ in sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:n]
    ]

    return recommendations


get_recommendations(rated_produnts, 5)



['B09PNR6F8Q', 'B082LZGK39', 'B07CRL2GY6', 'B09NHVCHS9', 'B08WRWPM22']

### Colaborative filtering

Train/Test split

In [12]:
ratings_df = ratings_df.drop_duplicates(
    subset=["user_id", "product_id"]
)

In [13]:
# remover duplicados (1464 → 1360 ratings únicos)
ratings_df = ratings_df.drop_duplicates(subset=["user_id", "product_id"])

# split por utilizador: users com ≥2 ratings têm 1 entrada no test set
train_list, test_list = [], []

for user_id, group in ratings_df.groupby("user_id"):
    if len(group) < 2:
        train_list.append(group)
    else:
        test_sample  = group.sample(n=1, random_state=42)
        train_sample = group.drop(test_sample.index)
        train_list.append(train_sample)
        test_list.append(test_sample)

train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

In [14]:
print("Total ratings :", len(ratings_df))
print("Train ratings :", len(train_df))
print("Test ratings  :", len(test_df))
print("Users dataset :", ratings_df["user_id"].nunique())
print("Users treino  :", train_df["user_id"].nunique())
print("Users teste   :", test_df["user_id"].nunique())

# confirmar que não há users no teste que não estão no treino
missing = set(test_df["user_id"]) - set(train_df["user_id"])
print("Users no teste sem histórico de treino:", len(missing))

Total ratings : 5858
Train ratings : 4667
Test ratings  : 1191
Users dataset : 1194
Users treino  : 1194
Users teste   : 1191
Users no teste sem histórico de treino: 0


In [15]:
users_train = set(train_df["user_id"])
users_test = set(test_df["user_id"])

missing_users = users_test - users_train

print("Users no teste que não existem no treino:", len(missing_users))

Users no teste que não existem no treino: 0


Criar user-item matrix com train

In [16]:
# user-item matrix (apenas dados de treino)
user_item_matrix        = train_df.pivot_table(
    index="user_id", columns="product_id", values="rating"
)
user_item_matrix_filled = user_item_matrix.fillna(0)

# item-item similarity
item_similarity    = cosine_similarity(user_item_matrix_filled.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns
)

# user-user similarity com mean-centering (reduz bias de utilizadores que
# tendem a dar ratings altos ou baixos sistematicamente)
user_means         = user_item_matrix.mean(axis=1)
user_item_centered = user_item_matrix.sub(user_means, axis=0).fillna(0)
user_similarity    = cosine_similarity(user_item_centered)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("user_item_matrix   :", user_item_matrix.shape)
print("item_similarity_df :", item_similarity_df.shape)
print("user_similarity_df :", user_similarity_df.shape)


user_item_matrix   : (1194, 1246)
item_similarity_df : (1246, 1246)
user_similarity_df : (1194, 1194)


Item-based Função de recomendações

In [17]:
def get_cf_recommendations(user_id, n=5):
    if user_id not in user_item_matrix.index:
        return []

    user_ratings = user_item_matrix.loc[user_id].dropna()

    scores = {}

    for product_id, rating in user_ratings.items():
        similar_items = item_similarity_df[product_id]

        for sim_product, sim_score in similar_items.items():
            if sim_product == product_id:
                continue

            scores[sim_product] = scores.get(sim_product, 0) + sim_score * rating

    # remover produtos já avaliados
    scores = {
        k: v for k, v in scores.items()
        if k not in user_ratings.index
    }

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [item for item, _ in ranked[:n]]

In [18]:
#testar
user_id = train_df["user_id"].iloc[0]

get_cf_recommendations(user_id)

['B08D77XZX5', 'B08GJNM9N7', 'B0B53QLB9H', 'B01D5H90L4', 'B09RWQ7YR6']

User-based Função de recomendações

In [19]:
# 2. Função de recomendação user-based
def get_user_based_cf_recommendations(user_id, n=5, k_users=10):
    """
    Para cada utilizador semelhante (top-k), agrega as suas ratings
    ponderadas pela similaridade. Exclui produtos já avaliados pelo user.
    """
    if user_id not in user_item_matrix.index:
        return []

    # Utilizadores mais semelhantes (excluindo o próprio)
    sim_users = (
        user_similarity_df[user_id]
        .drop(index=user_id)
        .sort_values(ascending=False)
        .head(k_users)
    )

    already_rated = set(
        user_item_matrix.loc[user_id].dropna().index
    )

    scores = {}

    for sim_user_id, sim_score in sim_users.items():
        # Produtos que este utilizador semelhante avaliou
        sim_user_ratings = user_item_matrix.loc[sim_user_id].dropna()

        for product_id, rating in sim_user_ratings.items():
            if product_id in already_rated:
                continue
            scores[product_id] = scores.get(product_id, 0) + sim_score * rating

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [product_id for product_id, _ in ranked[:n]]

In [20]:
# 3. Teste
user_id = train_df["user_id"].iloc[0]

print("User-based CF:", get_user_based_cf_recommendations(user_id))

User-based CF: ['B09HCH3JZG', 'B075ZTJ9XR', 'B08XXF5V6G', 'B09NNGHG22', 'B075TJHWVC']


Avaliar modelos collaborative filtering

In [21]:
def predict_rating_item_based(user_id, product_id, k=10):
    if user_id not in user_item_matrix.index:
        return None
    if product_id not in item_similarity_df.columns:
        return None

    user_ratings = user_item_matrix.loc[user_id].dropna()
    if user_ratings.empty:
        return None

    sim_scores = (
        item_similarity_df[product_id]
        .reindex(user_ratings.index)
        .dropna()
    )
    sim_scores = sim_scores[sim_scores > 0]
    if sim_scores.empty:
        return None

    top_k   = sim_scores.sort_values(ascending=False).head(k)
    ratings = user_ratings.reindex(top_k.index)
    denom   = top_k.sum()
    if denom == 0:
        return None
    return (top_k * ratings).sum() / denom


def predict_rating_user_based(user_id, product_id, k=10):
    if user_id not in user_similarity_df.index:
        return None
    if product_id not in user_item_matrix.columns:
        return None

    sim_users = (
        user_similarity_df[user_id]
        .drop(index=user_id)
        .sort_values(ascending=False)
    )
    sim_users = sim_users[sim_users > 0]
    if sim_users.empty:
        return None

    ratings_for_product = user_item_matrix[product_id].dropna()
    if ratings_for_product.empty:
        return None

    valid_users = sim_users.index.intersection(ratings_for_product.index)
    if len(valid_users) == 0:
        return None

    top_users = sim_users.loc[valid_users].head(k)
    ratings   = ratings_for_product.loc[top_users.index]
    denom     = top_users.sum()
    if denom == 0:
        return None
    return (top_users * ratings).sum() / denom


# ── loop de avaliação ──────────────────────────────────────────────────────────
item_preds, item_actuals = [], []
user_preds, user_actuals = [], []

for _, row in test_df.iterrows():
    uid, pid, actual = row["user_id"], row["product_id"], row["rating"]

    p_item = predict_rating_item_based(uid, pid)
    if p_item is not None:
        item_preds.append(p_item)
        item_actuals.append(actual)

    p_user = predict_rating_user_based(uid, pid)
    if p_user is not None:
        user_preds.append(p_user)
        user_actuals.append(actual)

# ── resultados ─────────────────────────────────────────────────────────────────
print("RESULTADOS")

if item_preds:
    rmse_item = np.sqrt(mean_squared_error(item_actuals, item_preds))
    print(f"Item-based CF  →  RMSE: {rmse_item:.4f}  ({len(item_preds)}/{len(test_df)} pares)")
else:
    print("Item-based CF  →  sem previsões suficientes.")

if user_preds:
    rmse_user = np.sqrt(mean_squared_error(user_actuals, user_preds))
    print(f"User-based CF  →  RMSE: {rmse_user:.4f}  ({len(user_preds)}/{len(test_df)} pares)")
else:
    print("User-based CF  →  0 previsões.")
    print("  Causa: dataset esparso (média 1.14 ratings/user).")
    print("  O modelo item-based é usado na integração do sistema.")


RESULTADOS
Item-based CF  →  RMSE: 0.8096  (1064/1191 pares)
User-based CF  →  RMSE: 0.8217  (727/1191 pares)


In [22]:
# ── Avaliação CF por Hit Rate@K ────────────────────────────────────────────────

def hit_rate_at_k(rec_fn, test_df, k=10):
    """
    Para cada user no test set, verifica se o item held-out
    aparece nas top-K recomendações geradas pelo modelo.
    """
    hits  = 0
    total = 0

    for _, row in test_df.iterrows():
        uid            = row["user_id"]
        held_out_item  = row["product_id"]

        recommendations = rec_fn(uid, n=k)

        if not recommendations:
            continue

        total += 1
        if held_out_item in recommendations:
            hits += 1

    if total == 0:
        return 0.0, 0

    return hits / total, total


# ── Resultados ─────────────────────────────────────────────────────────────────
for k in [5, 10, 20]:
    hr_item, n_item = hit_rate_at_k(get_cf_recommendations, test_df, k=k)
    hr_user, n_user = hit_rate_at_k(get_user_based_cf_recommendations, test_df, k=k)

    print(f"K={k:2d} | "
          f"Item-based Hit Rate: {hr_item:.4f} ({n_item} users) | "
          f"User-based Hit Rate: {hr_user:.4f} ({n_user} users)")

K= 5 | Item-based Hit Rate: 0.1830 (1191 users) | User-based Hit Rate: 0.0327 (1191 users)
K=10 | Item-based Hit Rate: 0.1914 (1191 users) | User-based Hit Rate: 0.0470 (1191 users)
K=20 | Item-based Hit Rate: 0.1973 (1191 users) | User-based Hit Rate: 0.0546 (1191 users)


In [23]:
# Verificar se as funções geram recomendações (ou retornam listas vazias)
sample_users = test_df["user_id"].iloc[:5]

for uid in sample_users:
    item_recs = get_cf_recommendations(uid, n=10)
    user_recs = get_user_based_cf_recommendations(uid, n=10)
    held_out  = test_df[test_df["user_id"] == uid]["product_id"].values[0]

    print(f"\nUser: {uid[:30]}...")
    print(f"  Held-out item : {held_out}")
    print(f"  Item-based recs ({len(item_recs)}): {item_recs[:3]}")
    print(f"  User-based recs ({len(user_recs)}): {user_recs[:3]}")


User: AE22Y3KIS7SE6LI3HE2VS6WWPU4Q,A...
  Held-out item : B00N3XLDW0
  Item-based recs (10): ['B08D77XZX5', 'B08GJNM9N7', 'B0B53QLB9H']
  User-based recs (10): ['B09HCH3JZG', 'B075ZTJ9XR', 'B08XXF5V6G']

User: AE23RS3W7GZO7LHYKJU6KSKVM4MQ,A...
  Held-out item : B009UORDX4
  Item-based recs (10): ['B091KNVNS9', 'B00H3H03Q4', 'B00SH18114']
  User-based recs (10): ['B09JN37WBX', 'B07JB2Y4SR', 'B0B25DJ352']

User: AE242TR3GQ6TYC6W4SJ5UYYKBTYQ...
  Held-out item : B00N3XLDW0
  Item-based recs (10): ['B0B15GSPQW', 'B09YL9SN9B', 'B081NHWT6Z']
  User-based recs (10): ['B0B15GSPQW', 'B0B16KD737', 'B08XXF5V6G']

User: AE27UOZENYSWCQVQRRUQIV2ZM7VA,A...
  Held-out item : B09V2PZDX8
  Item-based recs (10): ['B09GFM8CGS', 'B09GFPN6TP', 'B09GFPVD9Y']
  User-based recs (10): ['B09QS8V5N8', 'B09QS9CWLV', 'B09QS9X16F']

User: AE2JTMRKTUOIVIZWS2WDGTMNTU4Q,A...
  Held-out item : B084N133Y7
  Item-based recs (10): ['B07NTKGW45', 'B08FGNPQ9X', 'B08CTNJ985']
  User-based recs (10): ['B07NTKGW45', 'B07P681N6

In [24]:
# ── Preparar products_df para o Knowledge-Based RS ────────────────────────────
kbrs_df = products_df.copy()

kbrs_df['discounted_price']    = pd.to_numeric(kbrs_df['discounted_price'],    errors='coerce')
kbrs_df['actual_price']        = pd.to_numeric(kbrs_df['actual_price'],        errors='coerce')
kbrs_df['discount_percentage'] = pd.to_numeric(kbrs_df['discount_percentage'], errors='coerce')
kbrs_df['rating_count']        = pd.to_numeric(kbrs_df['rating_count'],        errors='coerce')

# rating médio vem do ratings_df
avg_ratings = (
    ratings_df.groupby('product_id')['rating']
    .mean()
    .reset_index()
    .rename(columns={'rating': 'avg_rating'})
)
kbrs_df = kbrs_df.merge(avg_ratings, on='product_id', how='left')
kbrs_df['avg_rating'] = pd.to_numeric(kbrs_df['avg_rating'], errors='coerce')

# top-level category
kbrs_df['top_category'] = kbrs_df['category'].str.split('|').str[0]

# score de popularidade
kbrs_df['popularity_score'] = (
    kbrs_df['avg_rating'] * np.log1p(kbrs_df['rating_count'])
)

print('Categorias disponíveis:')
print(kbrs_df['top_category'].value_counts().to_dict())
print(f'\nProdutos carregados: {len(kbrs_df)}')

Categorias disponíveis:
{'Electronics': 490, 'Home&Kitchen': 448, 'Computers&Accessories': 375, 'OfficeProducts': 31, 'MusicalInstruments': 2, 'HomeImprovement': 2, 'Toys&Games': 1, 'Car&Motorbike': 1, 'Health&PersonalCare': 1}

Produtos carregados: 1351


In [25]:
def knowledge_based_recommender(
    category       = None,   # ex: 'Electronics', 'Computers&Accessories', 'Home&Kitchen'
    max_price      = None,   # preço máximo em rupias
    min_rating     = 3.5,    # rating mínimo
    min_discount   = None,   # desconto mínimo ex: 0.20 = 20%
    min_rating_count = 100,  # número mínimo de avaliações
    n              = 5,      # número de recomendações
    user_age       = None,   # idade do utilizador (para regras demográficas)
    used_device    = None,   # 'Mobile', 'Desktop', 'Tablet'
):
    """
    Recommender baseado em restrições e regras.
    Filtra produtos pelo conjunto de restrições e ordena pelo
    popularity_score (rating × log(rating_count)).
    """
    df_filtered = kbrs_df.copy()

    # ── Regras demográficas automáticas ───────────────────────────────────────
    # Utilizadores jovens → aplicar desconto mínimo de 20% se não especificado
    if user_age is not None and user_age <= 30:
        if min_discount is None:
            min_discount = 0.20
            print("[Regra] Utilizador jovem (≤30) → desconto mínimo de 20% aplicado.")

    # Utilizadores Mobile → prioridade a categoria de telemóveis/acessórios
    if used_device == 'Mobile' and category is None:
        category = 'Computers&Accessories'
        print("[Regra] Utilizador Mobile → categoria 'Computers&Accessories' aplicada.")

    # Budget baixo → ordenar por desconto em vez de popularidade
    sort_by_discount = (max_price is not None and max_price < 500)

    # ── Restrições ─────────────────────────────────────────────────────────────
    if category is not None:
        df_filtered = df_filtered[
            df_filtered['top_category'].str.contains(category, case=False, na=False)
        ]

    if max_price is not None:
        df_filtered = df_filtered[df_filtered['discounted_price'] <= max_price]

    if min_rating is not None:
        df_filtered = df_filtered[df_filtered['avg_rating'] >= min_rating]

    if min_discount is not None:
        df_filtered = df_filtered[df_filtered['discount_percentage'] >= min_discount]

    if min_rating_count is not None:
        df_filtered = df_filtered[df_filtered['rating_count'] >= min_rating_count]

    # ── Ordenação ──────────────────────────────────────────────────────────────
    if df_filtered.empty:
        print("Nenhum produto encontrado com as restrições fornecidas.")
        return pd.DataFrame()

    if sort_by_discount:
        df_filtered = df_filtered.sort_values('popularity_score', ascending=False)
    else:
        df_filtered = df_filtered.sort_values('popularity_score', ascending=False)

    return df_filtered[[
        'product_id', 'product_name', 'top_category',
        'discounted_price', 'discount_percentage',
        'avg_rating', 'rating_count'
    ]].head(n).reset_index(drop=True)


In [26]:
# Teste 1 — Restrições básicas
# Utilizador quer Electronics, preço ≤ 1000, rating ≥ 4.0
print("=== Teste 1: Electronics, preço ≤ 1000, rating ≥ 4.0 ===")
knowledge_based_recommender(
    category='Electronics',
    max_price=1000,
    min_rating=4.0,
    n=5
)


=== Teste 1: Electronics, preço ≤ 1000, rating ≥ 4.0 ===


,product_id,product_name,top_category,discounted_price,discount_percentage,avg_rating,rating_count
0,B014I8SSD0,"Amazon Basics High-Speed HDMI Cable, 6 Feet - ...",Electronics,309.0,0.35,4.4,426973
1,B07KSMBL2H,AmazonBasics Flexible Premium HDMI Cable (Blac...,Electronics,219.0,0.69,4.4,426973
2,B014I8SX4Y,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",Electronics,309.0,0.78,4.4,426973
3,B09X7DY7Q4,SanDisk Extreme SD UHS I 64GB Card for 4K Vide...,Electronics,939.0,0.48,4.5,205052
4,B07GPXXNNG,boAt Bassheads 100 in Ear Wired Earphones with...,Electronics,349.0,0.65,4.1,363713


In [27]:
# Teste 2 — Regra demográfica: utilizador jovem (≤30)
# Desconto mínimo de 20% aplicado automaticamente
print("=== Teste 2: Utilizador jovem (age=25), budget 500 ===")
knowledge_based_recommender(
    max_price=500,
    min_rating=4.0,
    user_age=25,
    n=5
)


=== Teste 2: Utilizador jovem (age=25), budget 500 ===
[Regra] Utilizador jovem (≤30) → desconto mínimo de 20% aplicado.


,product_id,product_name,top_category,discounted_price,discount_percentage,avg_rating,rating_count
0,B014I8SSD0,"Amazon Basics High-Speed HDMI Cable, 6 Feet - ...",Electronics,309.0,0.35,4.4,426973
1,B07KSMBL2H,AmazonBasics Flexible Premium HDMI Cable (Blac...,Electronics,219.0,0.69,4.4,426973
2,B014I8SX4Y,"Amazon Basics High-Speed HDMI Cable, 6 Feet (2...",Electronics,309.0,0.78,4.4,426973
3,B005FYNT3G,SanDisk Cruzer Blade 32GB USB Flash Drive,Computers&Accessories,289.0,0.56,4.3,253105
4,B07GPXXNNG,boAt Bassheads 100 in Ear Wired Earphones with...,Electronics,349.0,0.65,4.1,363713


In [28]:
# Teste 3 — Regra demográfica: utilizador Mobile
# Categoria 'Computers&Accessories' aplicada automaticamente
print("=== Teste 3: Utilizador Mobile, preço ≤ 300 ===")
knowledge_based_recommender(
    max_price=300,
    min_rating=4.0,
    used_device='Mobile',
    n=5
)


=== Teste 3: Utilizador Mobile, preço ≤ 300 ===
[Regra] Utilizador Mobile → categoria 'Computers&Accessories' aplicada.


,product_id,product_name,top_category,discounted_price,discount_percentage,avg_rating,rating_count
0,B005FYNT3G,SanDisk Cruzer Blade 32GB USB Flash Drive,Computers&Accessories,289.000,0.56,4.3,253105
1,B00NH11KIK,AmazonBasics USB 2.0 Cable - A-Male to B-Male ...,Computers&Accessories,209.000,0.70,4.5,107687
2,B07G3YNLJB,Crucial BX500 240GB 3D NAND SATA 6.35 cm (2.5-...,Computers&Accessories,1.815,0.41,4.5,92925
3,B00NH13Q8W,AmazonBasics USB 2.0 Extension Cable for Perso...,Computers&Accessories,299.000,0.63,4.5,74977
4,B00NH11PEY,AmazonBasics USB 2.0 - A-Male to A-Female Exte...,Computers&Accessories,199.000,0.73,4.5,74976


In [29]:
# Teste 4 — Sem resultados (restrições muito apertadas)
print("=== Teste 4: Restrições muito apertadas ===")
knowledge_based_recommender(
    category='MusicalInstruments',
    max_price=100,
    min_rating=4.8,
    min_discount=0.5,
    n=5
)


=== Teste 4: Restrições muito apertadas ===
Nenhum produto encontrado com as restrições fornecidas.


""


In [30]:
# Teste 5 — Home&Kitchen com desconto mínimo de 30%
print("=== Teste 5: Home&Kitchen, desconto ≥ 30%, rating ≥ 4.0 ===")
knowledge_based_recommender(
    category='Home&Kitchen',
    min_rating=4.0,
    min_discount=0.30,
    n=5
)


=== Teste 5: Home&Kitchen, desconto ≥ 30%, rating ≥ 4.0 ===


,product_id,product_name,top_category,discounted_price,discount_percentage,avg_rating,rating_count
0,B01LWYDEQ7,Pigeon Polypropylene Mini Handy and Compact Ch...,Home&Kitchen,199.000,0.60,4.1,270563
1,B083GKDRKR,Havells Aqua Plus 1.2 litre Double Wall Kettle...,Home&Kitchen,1.625,0.46,4.5,23484
2,B00HVXS7WC,Bajaj Rex 500W Mixer Grinder with Nutri-Pro Fe...,Home&Kitchen,1.999,0.38,4.2,41349
3,B00EDJJ7FS,Philips Viva Collection HD4928/01 2100-Watt In...,Home&Kitchen,3.229,0.39,4.2,39724
4,B08Y5QJXSR,atomberg Renesa 1200mm BLDC Motor with Remote ...,Home&Kitchen,3.569,0.31,4.3,28629


### Hybrid Recommendation System

##### Switching Hybrid Recommendation System

In [31]:
def switching_hybrid_recommender(user_id, n=5):
    """
    Switching hybrid recommender:
      0 ratings  → non-personalized (top-rated products)
      1 rating   → content-based filtering
      >1 ratings → user-based collaborative filtering

    Returns: (DataFrame, strategy_name)
    """
    user_ratings = ratings_df[ratings_df["user_id"] == user_id]
    num_ratings = len(user_ratings)

    avg_ratings = ratings_df.groupby("product_id")["rating"].mean().rename("avg_rating")

    # ── 0 ratings: non-personalized ──────────────────────────────────────────
    if num_ratings == 0:
        strategy = "non_personalized"
        result = (
            merged_df
            .groupby(["product_id", "product_name", "category", "discounted_price"])["rating"]
            .mean()
            .reset_index(name="avg_rating")
            .sort_values("avg_rating", ascending=False)
            .head(n)
            .reset_index(drop=True)
        )
        result["strategy"] = strategy
        return result, strategy

    # ── 1 rating: content-based filtering ────────────────────────────────────
    elif num_ratings == 1:
        strategy = "content_based"
        rated = dict(zip(user_ratings["product_id"], user_ratings["rating"]))
        rec_ids = get_recommendations(rated, n)

    # ── >1 ratings: user-based collaborative filtering ────────────────────────
    else:
        strategy = "user_based_cf"
        rec_ids = get_user_based_cf_recommendations(user_id, n=n)

        # fallback to item-based if user-based returns nothing (sparse data)
        if not rec_ids:
            rec_ids = get_cf_recommendations(user_id, n=n)
            strategy = "item_based_cf_fallback"

    # ── Enrich product IDs with metadata ─────────────────────────────────────
    result = (
        products_df[products_df["product_id"].isin(rec_ids)]
        .set_index("product_id")
        .join(avg_ratings)
        [["product_name", "category", "discounted_price", "avg_rating"]]
        .reindex(rec_ids)  # preserve recommendation ranking order
        .reset_index()
    )
    result["strategy"] = strategy
    return result, strategy

In [32]:
# ── Test 1: 0 ratings → non-personalized ─────────────────────────────────────
unknown_user = "NEW_USER_WITH_NO_HISTORY"

recs, strategy = switching_hybrid_recommender(unknown_user, n=5)
print(f"Strategy: {strategy}  |  Ratings: 0")
recs

Strategy: non_personalized  |  Ratings: 0


,product_id,product_name,category,discounted_price,avg_rating,strategy
0,B0BQRJ3C47,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",Computers&Accessories|Accessories&Peripherals|...,249.000,5.0,non_personalized
1,B09ZHCJDP1,Amazon Basics Wireless Mouse | 2.4 GHz Connect...,Computers&Accessories|Accessories&Peripherals|...,499.000,5.0,non_personalized
2,B0BP7XLX48,Syncwire LTG to USB Cable for Fast Charging Co...,Computers&Accessories|Accessories&Peripherals|...,399.000,5.0,non_personalized
3,B0BR4F878Q,Swiffer Instant Electric Water Heater Faucet T...,"Home&Kitchen|Heating,Cooling&AirQuality|WaterH...",1.439,4.8,non_personalized
4,B0BQ3K23Y1,"Oratech Coffee Frother electric, milk frother ...",Home&Kitchen|Kitchen&HomeAppliances|SmallKitch...,279.000,4.8,non_personalized


In [33]:
# ── Test 2: exactly 1 rating → content-based filtering ───────────────────────
rating_counts = ratings_df.groupby("user_id").size()
single_rating_user = rating_counts[rating_counts == 1].index[0]

recs, strategy = switching_hybrid_recommender(single_rating_user, n=5)
rated_product = ratings_df[ratings_df["user_id"] == single_rating_user]["product_id"].values[0]
print(f"Strategy: {strategy}  |  Ratings: 1  |  Rated product: {rated_product}")
recs

Strategy: content_based  |  Ratings: 1  |  Rated product: B00DJ5N9VK


,product_id,product_name,category,discounted_price,avg_rating,strategy
0,B07SBGFDX9,"Pentonic Multicolor Ball Point Pen, Pack of 10",OfficeProducts|OfficePaperProducts|Paper|Stati...,120.000,4.329412,content_based
1,B0746N6WML,Parker Vector Camouflage Gift Set - Roller Bal...,OfficeProducts|OfficePaperProducts|Paper|Stati...,341.000,4.300000,content_based
2,B00LVMTA2A,Panasonic CR-2032/5BE Lithium Coin Battery - P...,Electronics|GeneralPurposeBatteries&BatteryCha...,225.000,4.400000,content_based
3,B0B8VQ7KDS,Airtel Digital TV HD Set Top Box with FTA Pack...,"Electronics|HomeTheater,TV&Video|SatelliteEqui...",1.299,4.300000,content_based
4,B00S2SEV7K,"Pilot Frixion Clicker Roller Pen (Blue), (9000...",OfficeProducts|OfficePaperProducts|Paper|Stati...,90.000,3.975000,content_based


In [34]:
# ── Test 3: >1 ratings → user-based collaborative filtering ──────────────────
multi_rating_user = rating_counts[rating_counts > 1].index[0]
num = rating_counts[multi_rating_user]

recs, strategy = switching_hybrid_recommender(multi_rating_user, n=5)
print(f"Strategy: {strategy}  |  Ratings: {num}")
recs

Strategy: user_based_cf  |  Ratings: 5


,product_id,product_name,category,discounted_price,avg_rating,strategy
0,B09HCH3JZG,Bestor ® 8K Hdmi 2.1 Cable 48Gbps 9.80Ft/Ultra...,"Electronics|HomeTheater,TV&Video|Accessories|C...",699.000,4.4,user_based_cf
1,B075ZTJ9XR,AmazonBasics High-Speed Braided HDMI Cable - 3...,"Electronics|HomeTheater,TV&Video|Accessories|C...",269.000,4.4,user_based_cf
2,B08XXF5V6G,Kodak 139 cm (55 inches) 4K Ultra HD Smart LED...,"Electronics|HomeTheater,TV&Video|Televisions|S...",29.999,4.4,user_based_cf
3,B09NNGHG22,Sansui 140cm (55 inches) 4K Ultra HD Certified...,"Electronics|HomeTheater,TV&Video|Televisions|S...",32.990,4.3,user_based_cf
4,B075TJHWVC,Airtel Digital TV HD Set Top Box with 1 Month ...,"Electronics|HomeTheater,TV&Video|SatelliteEqui...",917.000,4.2,user_based_cf


In [35]:
# ── Summary: strategy distribution across all users ──────────────────────────
print("Rating count distribution among users:")
print(rating_counts.value_counts().sort_index().rename("num_users").to_frame().head(10))

cold  = (rating_counts == 0).sum() + users_df[~users_df["user_id"].isin(ratings_df["user_id"])].shape[0]
cbf   = (rating_counts == 1).sum()
ubcf  = (rating_counts  > 1).sum()
total = cold + cbf + ubcf

print(f"\nSwitching strategy allocation:")
print(f"  non_personalized  (0 ratings) : {cold:>4}  ({cold/total*100:.1f}%)")
print(f"  content_based     (1 rating)  : {cbf:>4}  ({cbf/total*100:.1f}%)")
print(f"  user_based_cf     (>1 ratings): {ubcf:>4}  ({ubcf/total*100:.1f}%)")

Rating count distribution among users:
   num_users
1          3
2          4
3         17
4         62
5       1104
6          2
8          2

Switching strategy allocation:
  non_personalized  (0 ratings) :    0  (0.0%)
  content_based     (1 rating)  :    3  (0.3%)
  user_based_cf     (>1 ratings): 1191  (99.7%)


##### Mixed Hybrid Recommendation System

In [36]:
# Pre-build product index map once for fast similarity lookups
_product_index_map = pd.Series(
    products_df.index, index=products_df["product_id"]
).to_dict()


def _find_cbf_source(rec_id, rated_dict):
    """Return the rated product_id most similar (content-wise) to rec_id."""
    if rec_id not in _product_index_map:
        return None
    rec_idx = _product_index_map[rec_id]
    best_pid, best_score = None, -1.0
    for pid in rated_dict:
        if pid not in _product_index_map:
            continue
        score = float(similarity_matrix[rec_idx, _product_index_map[pid]])
        if score > best_score:
            best_score, best_pid = score, pid
    return best_pid


def _product_row(product_id, avg_ratings):
    row = products_df[products_df["product_id"] == product_id]
    if row.empty:
        return None
    r = row.iloc[0]
    return {
        "product_id":       product_id,
        "product_name":     r["product_name"],
        "category":         r["category"].split("|")[0],
        "discounted_price": r["discounted_price"],
        "avg_rating":       round(avg_ratings.get(product_id, float("nan")), 2),
    }


def _short(name, limit=45):
    return name if len(name) <= limit else name[:limit].rstrip() + "…"


def mixed_hybrid_recommender(user_id, n_top=3, n_cbf=3, n_cf=4):
    """
    Mixed hybrid recommender — always combines three sources:
      · n_top  non-personalized  →  top-rated products
      · n_cbf  content-based     →  "because you liked <X>"
      · n_cf   user-based CF     →  "users like you also liked…"

    When a user lacks ratings the missing slices fall back to top-rated
    so the total always reaches n_top + n_cbf + n_cf recommendations.
    """
    user_ratings_df = ratings_df[ratings_df["user_id"] == user_id]
    num_ratings     = len(user_ratings_df)
    avg_ratings     = ratings_df.groupby("product_id")["rating"].mean()
    seen            = set()
    results         = []

    # ── helper: pull top-rated candidates (used as fallback too) ─────────────
    top_pool = (
        merged_df
        .groupby(["product_id", "product_name", "category", "discounted_price"])["rating"]
        .mean()
        .reset_index(name="avg_rating")
        .sort_values("avg_rating", ascending=False)
    )

    def _take_from_top(n, source_label, explanation_fn):
        count = 0
        for _, row in top_pool.iterrows():
            if count >= n:
                break
            if row["product_id"] in seen:
                continue
            seen.add(row["product_id"])
            results.append({
                "product_id":       row["product_id"],
                "product_name":     row["product_name"],
                "category":         row["category"].split("|")[0],
                "discounted_price": row["discounted_price"],
                "avg_rating":       round(row["avg_rating"], 2),
                "source":           source_label,
                "explanation":      explanation_fn(row),
            })
            count += 1

    # ── Slice 1: Non-personalized ─────────────────────────────────────────────
    _take_from_top(
        n_top,
        source_label   = "non_personalized",
        explanation_fn = lambda r: f"⭐ Top rated product (avg {r['avg_rating']:.1f})",
    )

    # ── Slice 2: Content-based filtering ─────────────────────────────────────
    if num_ratings >= 1:
        rated = dict(zip(user_ratings_df["product_id"], user_ratings_df["rating"]))
        cbf_ids = get_recommendations(rated, n_cbf * 4)   # fetch extra to cover dedup losses

        count = 0
        for rec_id in cbf_ids:
            if count >= n_cbf:
                break
            if rec_id in seen:
                continue
            info = _product_row(rec_id, avg_ratings)
            if info is None:
                continue
            seen.add(rec_id)

            source_pid  = _find_cbf_source(rec_id, rated)
            source_name = _short(
                products_df[products_df["product_id"] == source_pid]["product_name"].values[0]
            ) if source_pid else "a product you rated"

            info["source"]      = "content_based"
            info["explanation"] = f'🔍 Because you liked "{source_name}"'
            results.append(info)
            count += 1

        # fallback: fill remaining CBF slots with top-rated
        if count < n_cbf:
            _take_from_top(
                n_cbf - count,
                source_label   = "non_personalized_fallback",
                explanation_fn = lambda r: f"⭐ Top rated product (avg {r['avg_rating']:.1f})",
            )
    else:
        # cold-start: fill CBF slots with top-rated
        _take_from_top(
            n_cbf,
            source_label   = "non_personalized_fallback",
            explanation_fn = lambda r: f"⭐ Popular pick — rate products to get personalised suggestions",
        )

    # ── Slice 3: User-based collaborative filtering ───────────────────────────
    if num_ratings > 1:
        cf_ids = get_user_based_cf_recommendations(user_id, n=n_cf * 4)
        cf_source = "user_based_cf"
        if not cf_ids:                            # sparse fallback
            cf_ids    = get_cf_recommendations(user_id, n=n_cf * 4)
            cf_source = "item_based_cf"

        count = 0
        for rec_id in cf_ids:
            if count >= n_cf:
                break
            if rec_id in seen:
                continue
            info = _product_row(rec_id, avg_ratings)
            if info is None:
                continue
            seen.add(rec_id)

            info["source"]      = cf_source
            info["explanation"] = "👥 Users with similar taste also liked this"
            results.append(info)
            count += 1

        if count < n_cf:
            _take_from_top(
                n_cf - count,
                source_label   = "non_personalized_fallback",
                explanation_fn = lambda r: f"⭐ Top rated product (avg {r['avg_rating']:.1f})",
            )
    else:
        msg = (
            "Rate 1 more product to unlock personalised CF recommendations"
            if num_ratings == 1
            else "Rate some products to unlock personalised recommendations"
        )
        _take_from_top(
            n_cf,
            source_label   = "non_personalized_fallback",
            explanation_fn = lambda r: f"⭐ {msg}",
        )

    df = pd.DataFrame(results, columns=[
        "product_id", "product_name", "category",
        "discounted_price", "avg_rating", "source", "explanation"
    ])
    df.index = range(1, len(df) + 1)
    return df

In [37]:
# ── Test A: cold-start user (0 ratings) ──────────────────────────────────────
recs_cold = mixed_hybrid_recommender("BRAND_NEW_USER")
print("=== Cold start (0 ratings) — all 10 slots filled by top-rated fallback ===\n")
print(recs_cold[["product_name", "source", "explanation"]].to_string())
print(f"\nTotal: {len(recs_cold)} recommendations")

=== Cold start (0 ratings) — all 10 slots filled by top-rated fallback ===

                                                                                                                                                                                                                                                                    product_name                     source                                                     explanation
1                                                                                  REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi Certified] Lightning to Type C Fast Charging Cord Compatible with iPhone 14/13/13 pro/Max/12/11/X/XS/XR/8, Supports Power Delivery - White           non_personalized                                   ⭐ Top rated product (avg 5.0)
2                                                                                                             Amazon Basics Wireless Mouse | 2.4 GHz Connection, 1600 DPI | Type - C Adapter | Upto 

In [38]:
# ── Test B: 1-rating user → CBF slice active, CF still fallback ──────────────
rating_counts = ratings_df.groupby("user_id").size()
single_user   = rating_counts[rating_counts == 1].index[0]
rated_pid     = ratings_df[ratings_df["user_id"] == single_user]["product_id"].values[0]
rated_name    = products_df[products_df["product_id"] == rated_pid]["product_name"].values[0]

print(f"=== 1-rating user ===")
print(f"Rated: {rated_name[:60]}\n")

recs_single = mixed_hybrid_recommender(single_user)
print(recs_single[["product_name", "source", "explanation"]].to_string())
print(f"\nTotal: {len(recs_single)} recommendations")

=== 1-rating user ===
Rated: Faber-Castell Connector Pen Set - Pack of 25 (Assorted)

                                                                                                                                                                                                                                                                    product_name                     source                                                          explanation
1                                                                                  REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi Certified] Lightning to Type C Fast Charging Cord Compatible with iPhone 14/13/13 pro/Max/12/11/X/XS/XR/8, Supports Power Delivery - White           non_personalized                                        ⭐ Top rated product (avg 5.0)
2                                                                                                             Amazon Basics Wireless Mouse | 2.4 GHz Connection, 1600 DPI | Type

In [39]:
# ── Test C: >1 ratings → all three slices fully active ───────────────────────
multi_user  = rating_counts[rating_counts > 1].index[0]
num_ratings = rating_counts[multi_user]

print(f"=== Full mixed hybrid ({num_ratings} ratings) ===\n")

recs_full = mixed_hybrid_recommender(multi_user)
print(recs_full[["product_name", "source", "explanation"]].to_string())
print(f"\nTotal: {len(recs_full)} recommendations")
print(f"\nSource breakdown:\n{recs_full['source'].value_counts().to_string()}")

=== Full mixed hybrid (5 ratings) ===

                                                                                                                                                                                      product_name            source                                                           explanation
1    REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi Certified] Lightning to Type C Fast Charging Cord Compatible with iPhone 14/13/13 pro/Max/12/11/X/XS/XR/8, Supports Power Delivery - White  non_personalized                                         ⭐ Top rated product (avg 5.0)
2                               Amazon Basics Wireless Mouse | 2.4 GHz Connection, 1600 DPI | Type - C Adapter | Upto 12 Months of Battery Life | Ambidextrous Design | Suitable for PC/Mac/Laptop  non_personalized                                         ⭐ Top rated product (avg 5.0)
3                  Syncwire LTG to USB Cable for Fast Charging Compatible with Phone 5/ 5C/ 5S/ 6/ 6S/ 7/8/ X/XR

##### Weighted Hybrid Recommender

In [40]:
def _normalize(series):
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn + 1e-9)


def weighted_hybrid_recommender(user_id, n=10, w_np=0.2, w_cbf=0.4, w_cf=0.4):
    """
    Weighted hybrid: every candidate product is scored by all three models,
    scores are normalized to [0,1], then combined as a weighted sum.

      final_score = w_np * np_score + w_cbf * cbf_score + w_cf * cf_score

    Weights are auto-adjusted when data is missing:
      0 ratings  → pure non-personalized  (w_np=1)
      1 rating   → NP + CBF only          (weights renormalized)
      >1 ratings → all three active       (use supplied weights)

    Returns a DataFrame with individual score columns so you can inspect
    each model's contribution to every recommendation.
    """
    user_ratings_df = ratings_df[ratings_df["user_id"] == user_id]
    num_ratings     = len(user_ratings_df)
    avg_ratings     = ratings_df.groupby("product_id")["rating"].mean()
    already_rated   = set(user_ratings_df["product_id"])
    candidates      = [p for p in avg_ratings.index if p not in already_rated]

    # ── Score 1: Non-personalized (avg_rating, normalized) ───────────────────
    np_scores = _normalize(avg_ratings.reindex(candidates).fillna(0))

    # ── Score 2: Content-based (vectorized cosine similarity) ────────────────
    cbf_scores = pd.Series(0.0, index=candidates)
    if num_ratings >= 1:
        rated       = dict(zip(user_ratings_df["product_id"], user_ratings_df["rating"]))
        cbf_vector  = np.zeros(len(products_df))
        for pid, r in rated.items():
            if pid in _product_index_map:
                cbf_vector += similarity_matrix[:, _product_index_map[pid]] * r
        cand_idxs  = [_product_index_map.get(p) for p in candidates]
        raw_cbf    = pd.Series(
            [cbf_vector[i] if i is not None else 0.0 for i in cand_idxs],
            index=candidates
        )
        cbf_scores = _normalize(raw_cbf)

    # ── Score 3: Item-based CF (vectorized dot product) ───────────────────────
    cf_scores = pd.Series(0.0, index=candidates)
    if num_ratings > 1 and user_id in user_item_matrix.index:
        user_rat  = user_item_matrix.loc[user_id].dropna()
        valid     = [p for p in candidates if p in item_similarity_df.index]
        raw_cf    = (
            item_similarity_df
            .reindex(index=valid, columns=user_rat.index)
            .fillna(0) @ user_rat
        )
        cf_scores = _normalize(raw_cf.reindex(candidates).fillna(0))

    # ── Adapt weights to available data ──────────────────────────────────────
    if num_ratings == 0:
        w1, w2, w3 = 1.0, 0.0, 0.0
    elif num_ratings == 1:
        total       = w_np + w_cbf
        w1, w2, w3  = w_np / total, w_cbf / total, 0.0
    else:
        w1, w2, w3  = w_np, w_cbf, w_cf

    final_scores = (
        w1 * np_scores +
        w2 * cbf_scores.reindex(candidates).fillna(0) +
        w3 * cf_scores.reindex(candidates).fillna(0)
    )

    top_ids = final_scores.nlargest(n).index.tolist()

    rows = []
    for pid in top_ids:
        info = _product_row(pid, avg_ratings)
        if info is None:
            continue
        info["np_score"]    = round(float(np_scores.get(pid, 0)),    4)
        info["cbf_score"]   = round(float(cbf_scores.get(pid, 0)),   4)
        info["cf_score"]    = round(float(cf_scores.get(pid, 0)),    4)
        info["final_score"] = round(float(final_scores.get(pid, 0)), 4)
        rows.append(info)

    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    return df

In [48]:
# ── Test: full-data user with default weights (0.2 / 0.4 / 0.4) ──────────────
multi_user = rating_counts[rating_counts > 1].index[0]

recs_w = weighted_hybrid_recommender(multi_user, n=10)
print(f"=== Weighted hybrid — user with {rating_counts[multi_user]} ratings ===\n")
print(recs_w[["product_name", "np_score", "cbf_score", "cf_score", "final_score"]].to_string())

=== Weighted hybrid — user with 5 ratings ===

                                                                                                                                                                                               product_name  np_score  cbf_score  cf_score  final_score
1                                                                                                               LOHAYA Television Remote Compatible for VU LED LCD HD Tv Remote Control Model No :- EN2B27V    0.6756     0.9867    0.9040       0.8914
2   PTron Tangentbeat in-Ear Bluetooth 5.0 Wireless Headphones with Mic, Enhanced Bass, 10mm Drivers, Clear Calls, Snug-Fit, Fast Charging, Magnetic Buds, Voice Assistant & IPX4 Wireless Neckband (Black)    0.6517     0.6070    1.0000       0.7731
3                                                                                        Caldipree Silicone Case Cover Compatible for 2022 Samsung Smart TV Remote QLED TV BN68-13897A TM2280E (2022-BLACK)    0.

In [42]:
# ── Weight sensitivity: how do top-5 products change across configurations? ───
configs = {
    "NP-heavy   (0.8/0.1/0.1)": (0.8, 0.1, 0.1),
    "CBF-heavy  (0.1/0.8/0.1)": (0.1, 0.8, 0.1),
    "CF-heavy   (0.1/0.1/0.8)": (0.1, 0.1, 0.8),
    "Balanced   (0.33/0.33/0.33)": (0.33, 0.33, 0.33),
    "Default    (0.2/0.4/0.4)": (0.2, 0.4, 0.4),
}

for label, (w1, w2, w3) in configs.items():
    recs = weighted_hybrid_recommender(multi_user, n=5, w_np=w1, w_cbf=w2, w_cf=w3)
    top5 = recs["product_name"].apply(lambda x: x[:50]).tolist()
    print(f"\n{label}")
    for i, name in enumerate(top5, 1):
        print(f"  {i}. {name}")


NP-heavy   (0.8/0.1/0.1)
  1. Syncwire LTG to USB Cable for Fast Charging Compat
  2. REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi
  3. Amazon Basics Wireless Mouse | 2.4 GHz Connection,
  4. Sony Bravia 164 cm (65 inches) 4K Ultra HD Smart L
  5. 10k 8k 4k HDMI Cable, Certified 48Gbps 1ms Ultra H

CBF-heavy  (0.1/0.8/0.1)
  1. LOHAYA Television Remote Compatible for VU LED LCD
  2. 7SEVEN® Compatible for Sony Bravia LCD LED UHD OLE
  3. 7SEVEN® Compatible for Tata Sky Remote Original Se
  4. 7SEVEN® Suitable Sony Tv Remote Original Bravia fo
  5. Cotbolt Silicone Protective Case Cover for LG an M

CF-heavy   (0.1/0.1/0.8)
  1. PTron Tangentbeat in-Ear Bluetooth 5.0 Wireless He
  2. LOHAYA Television Remote Compatible for VU LED LCD
  3. PTron Newly Launched Force X10 Bluetooth Calling S
  4. AmazonBasics - High-Speed Male to Female HDMI Exte
  5. MI 138.8 cm (55 inches) 5X Series 4K Ultra HD LED 

Balanced   (0.33/0.33/0.33)
  1. LOHAYA Television Remote Compatible for VU LED L

##### Cascade Recommender

In [43]:
def cascade_recommender(user_id, n=10, pool_1=60, pool_2=30):
    """
    Cascade (funnel) recommender — output of each stage feeds the next:

      Stage 1  Non-personalized  top pool_1 products by avg_rating
          ↓
      Stage 2  Content-based     re-rank pool_1 by CBF score → keep top pool_2
          ↓
      Stage 3  Item-based CF     re-rank pool_2 by CF score  → return top n

    When data is insufficient, stages pass their input through unchanged
    so the funnel still produces n results.

    Output includes np_rank / cbf_rank / cf_rank so you can trace how
    each product moved through the pipeline.
    """
    user_ratings_df = ratings_df[ratings_df["user_id"] == user_id]
    num_ratings     = len(user_ratings_df)
    avg_ratings     = ratings_df.groupby("product_id")["rating"].mean()
    already_rated   = set(user_ratings_df["product_id"])

    # ── Stage 1: Non-personalized candidate pool ──────────────────────────────
    stage1_series = (
        avg_ratings[~avg_ratings.index.isin(already_rated)]
        .sort_values(ascending=False)
        .head(pool_1)
    )
    stage1_ids = stage1_series.index.tolist()
    np_rank    = {pid: i + 1 for i, pid in enumerate(stage1_ids)}

    # ── Stage 2: CBF re-ranking ───────────────────────────────────────────────
    cbf_rank = {}
    if num_ratings >= 1:
        rated       = dict(zip(user_ratings_df["product_id"], user_ratings_df["rating"]))
        cbf_vector  = np.zeros(len(products_df))
        for pid, r in rated.items():
            if pid in _product_index_map:
                cbf_vector += similarity_matrix[:, _product_index_map[pid]] * r

        cbf_raw = pd.Series({
            pid: cbf_vector[_product_index_map[pid]] if pid in _product_index_map else 0.0
            for pid in stage1_ids
        })
        stage2_ids = cbf_raw.sort_values(ascending=False).head(pool_2).index.tolist()
        cbf_rank   = {pid: i + 1 for i, pid in enumerate(stage2_ids)}
    else:
        stage2_ids = stage1_ids[:pool_2]   # pass-through: no CBF data

    # ── Stage 3: CF re-ranking ────────────────────────────────────────────────
    cf_rank = {}
    if num_ratings > 1 and user_id in user_item_matrix.index:
        user_rat = user_item_matrix.loc[user_id].dropna()
        valid    = [p for p in stage2_ids if p in item_similarity_df.index]
        raw_cf   = (
            item_similarity_df
            .reindex(index=valid, columns=user_rat.index)
            .fillna(0) @ user_rat
        )
        raw_cf     = raw_cf.reindex(stage2_ids).fillna(0)
        final_ids  = raw_cf.sort_values(ascending=False).head(n).index.tolist()
        cf_rank    = {pid: i + 1 for i, pid in enumerate(final_ids)}
    else:
        final_ids = stage2_ids[:n]         # pass-through: no CF data

    # ── Build output with rank journey ────────────────────────────────────────
    rows = []
    for pid in final_ids:
        info = _product_row(pid, avg_ratings)
        if info is None:
            continue
        info["np_rank"]  = np_rank.get(pid, "—")
        info["cbf_rank"] = cbf_rank.get(pid, "—") if num_ratings >= 1 else "—"
        info["cf_rank"]  = cf_rank.get(pid,  "—") if num_ratings >  1 else "—"

        # rank delta: positive = moved up through the pipeline
        np_r  = np_rank.get(pid, pool_1)
        fin_r = list(final_ids).index(pid) + 1
        info["rank_delta"] = np_r - fin_r   # +N = rose N places vs NP baseline

        rows.append(info)

    df = pd.DataFrame(rows)
    df.index = range(1, len(df) + 1)
    return df

In [44]:
# ── Test cascade: full-data user — all 3 stages active ───────────────────────
multi_user = rating_counts[rating_counts > 1].index[0]

recs_c = cascade_recommender(multi_user, n=10, pool_1=60, pool_2=30)
print(f"=== Cascade recommender — user with {rating_counts[multi_user]} ratings ===")
print(f"Funnel: 60 (NP) → 30 (CBF) → 10 (CF)\n")
print(recs_c[["product_name", "np_rank", "cbf_rank", "cf_rank", "rank_delta"]].to_string())
print("\n rank_delta > 0 = product rose in the pipeline vs NP baseline")

=== Cascade recommender — user with 5 ratings ===
Funnel: 60 (NP) → 30 (CBF) → 10 (CF)

                                                                                                                                                                                                product_name  np_rank  cbf_rank  cf_rank  rank_delta
1                                                                                                                     AmazonBasics 10.2 Gbps High-Speed 4K HDMI Cable with Braided Cord (10-Foot, Dark Grey)       55         2        1          54
2                                                                                                                           Sony Bravia 164 cm (65 inches) 4K Ultra HD Smart LED Google TV KD-65X74K (Black)        7         3        2           5
3     WANBO X1 Pro (Upgraded) | Native 1080P Full HD | Android 9 | Projector for Home | LED Cinema | 350ANSI | 3900 lumens | WiFi Bluetooth | HDMI ARC | Dolby DTS | 4D Keystone 

In [45]:
# ── Compare all four hybrid approaches side-by-side (top-5 product names) ────
user = rating_counts[rating_counts > 1].index[0]

approaches = {
    "Switching":          switching_hybrid_recommender(user, n=5)[0]["product_name"].tolist(),
    "Mixed":              mixed_hybrid_recommender(user, n_top=2, n_cbf=2, n_cf=1)["product_name"].tolist(),
    "Weighted (default)": weighted_hybrid_recommender(user, n=5)["product_name"].tolist(),
    "Cascade":            cascade_recommender(user, n=5)["product_name"].tolist(),
}

print("=== Top-5 comparison across hybrid approaches ===\n")
for approach, names in approaches.items():
    print(f"{approach}:")
    for i, name in enumerate(names, 1):
        print(f"  {i}. {name[:70]}")
    print()

=== Top-5 comparison across hybrid approaches ===

Switching:
  1. Bestor ® 8K Hdmi 2.1 Cable 48Gbps 9.80Ft/Ultra High Speed Hdmi Braided
  2. AmazonBasics High-Speed Braided HDMI Cable - 3 Feet - Supports Etherne
  3. Kodak 139 cm (55 inches) 4K Ultra HD Smart LED TV 55CA0909 (Black)
  4. Sansui 140cm (55 inches) 4K Ultra HD Certified Android LED TV with Dol
  5. Airtel Digital TV HD Set Top Box with 1 Month Basic Pack with Recordin

Mixed:
  1. REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi Certified] Lightnin
  2. Amazon Basics Wireless Mouse | 2.4 GHz Connection, 1600 DPI | Type - C
  3. Samsung 138 cm (55 inches) Crystal 4K Series Ultra HD Smart LED TV UA5
  4. SanDisk Ultra® microSDXC™ UHS-I Card, 128GB, 140MB/s R, 10 Y Warranty,
  5. Bestor ® 8K Hdmi 2.1 Cable 48Gbps 9.80Ft/Ultra High Speed Hdmi Braided

Weighted (default):
  1. LOHAYA Television Remote Compatible for VU LED LCD HD Tv Remote Contro
  2. PTron Tangentbeat in-Ear Bluetooth 5.0 Wireless Headphones with Mic, E
 

### Comprehensive Evaluation — All Recommenders

In [49]:
import time

# Pre-computed globals reused by all eval wrappers
_avg_ratings_eval = ratings_df.groupby('product_id')['rating'].mean()
_top_all_ids      = _avg_ratings_eval.sort_values(ascending=False).index.tolist()

def _ndcg(recs, held_out, k):
    top_k = recs[:k]
    if held_out not in top_k:
        return 0.0
    return 1.0 / np.log2(top_k.index(held_out) + 2)

# ── Evaluation-safe wrappers (use train_df so the held-out item is a valid candidate) ──
def eval_np(user_id, n):
    train_rated = set(train_df[train_df['user_id'] == user_id]['product_id'])
    return [p for p in _top_all_ids if p not in train_rated][:n]

def eval_cbf(user_id, n):
    ut = train_df[train_df['user_id'] == user_id]
    if ut.empty: return []
    rated = dict(zip(ut['product_id'], ut['rating']))
    return get_recommendations(rated, n)

def eval_item_cf(user_id, n):
    return get_cf_recommendations(user_id, n)

def eval_user_cf(user_id, n):
    return get_user_based_cf_recommendations(user_id, n)

def eval_switching(user_id, n):
    num = len(train_df[train_df['user_id'] == user_id])
    if num == 0:   return eval_np(user_id, n)
    elif num == 1: return eval_cbf(user_id, n)
    else:
        r = eval_user_cf(user_id, n)
        return r if r else eval_item_cf(user_id, n)

def eval_mixed(user_id, n):
    ut     = train_df[train_df['user_id'] == user_id]
    num    = len(ut)
    seen   = set(ut['product_id'])
    result = []
    for p in _top_all_ids:                      # NP slice (3)
        if p not in seen: seen.add(p); result.append(p)
        if len(result) >= 3: break
    if num >= 1:
        rated = dict(zip(ut['product_id'], ut['rating']))
        for p in get_recommendations(rated, 12):  # CBF slice (3)
            if p not in seen: seen.add(p); result.append(p)
            if len(result) >= 6: break
    for p in eval_item_cf(user_id, 16):          # CF slice (4)
        if p not in seen: seen.add(p); result.append(p)
        if len(result) >= 10: break
    return result[:n]

def eval_weighted(user_id, n, w_np=0.2, w_cbf=0.4, w_cf=0.4):
    ut          = train_df[train_df['user_id'] == user_id]
    num         = len(ut)
    train_rated = set(ut['product_id'])
    candidates  = [p for p in _avg_ratings_eval.index if p not in train_rated]
    np_s  = _normalize(_avg_ratings_eval.reindex(candidates).fillna(0))
    cbf_s = pd.Series(0.0, index=candidates)
    cf_s  = pd.Series(0.0, index=candidates)
    if num >= 1:
        rated   = dict(zip(ut['product_id'], ut['rating']))
        cbf_vec = np.zeros(len(products_df))
        for pid, r in rated.items():
            if pid in _product_index_map:
                cbf_vec += similarity_matrix[:, _product_index_map[pid]] * r
        raw_cbf = pd.Series(
            [cbf_vec[_product_index_map[p]] if p in _product_index_map else 0.0 for p in candidates],
            index=candidates)
        cbf_s = _normalize(raw_cbf)
    if num > 1 and user_id in user_item_matrix.index:
        user_rat = user_item_matrix.loc[user_id].dropna()
        valid    = [p for p in candidates if p in item_similarity_df.index]
        raw_cf   = item_similarity_df.reindex(index=valid, columns=user_rat.index).fillna(0) @ user_rat
        cf_s     = _normalize(raw_cf.reindex(candidates).fillna(0))
    if num == 0:   w1, w2, w3 = 1.0, 0.0, 0.0
    elif num == 1: w1, w2, w3 = w_np/(w_np+w_cbf), w_cbf/(w_np+w_cbf), 0.0
    else:          w1, w2, w3 = w_np, w_cbf, w_cf
    final = w1*np_s + w2*cbf_s.reindex(candidates).fillna(0) + w3*cf_s.reindex(candidates).fillna(0)
    return final.nlargest(n).index.tolist()

def eval_cascade(user_id, n, pool_1=60, pool_2=30):
    ut          = train_df[train_df['user_id'] == user_id]
    num         = len(ut)
    train_rated = set(ut['product_id'])
    stage1 = (_avg_ratings_eval[~_avg_ratings_eval.index.isin(train_rated)]
               .sort_values(ascending=False).head(pool_1).index.tolist())
    if num >= 1:
        rated   = dict(zip(ut['product_id'], ut['rating']))
        cbf_vec = np.zeros(len(products_df))
        for pid, r in rated.items():
            if pid in _product_index_map:
                cbf_vec += similarity_matrix[:, _product_index_map[pid]] * r
        cbf_raw = pd.Series({pid: cbf_vec[_product_index_map[pid]] if pid in _product_index_map else 0.0 for pid in stage1})
        stage2  = cbf_raw.sort_values(ascending=False).head(pool_2).index.tolist()
    else:
        stage2 = stage1[:pool_2]
    if num > 1 and user_id in user_item_matrix.index:
        user_rat = user_item_matrix.loc[user_id].dropna()
        valid    = [p for p in stage2 if p in item_similarity_df.index]
        raw_cf   = item_similarity_df.reindex(index=valid, columns=user_rat.index).fillna(0) @ user_rat
        return raw_cf.reindex(stage2).fillna(0).sort_values(ascending=False).head(n).index.tolist()
    return stage2[:n]

print('Evaluation wrappers ready.')


Evaluation wrappers ready.


In [50]:
def run_evaluation(rec_fns, test_df, k_values=(5, 10, 20)):
    all_items = set(ratings_df['product_id'].unique())
    results   = {}
    for name, fn in rec_fns.items():
        print(f'  {name:<28}', end=' ', flush=True)
        t0         = time.time()
        hits       = {k: 0   for k in k_values}
        ndcg_sum   = {k: 0.0 for k in k_values}
        recs_union = set()
        total      = 0
        for _, row in test_df.iterrows():
            uid, held_out = row['user_id'], row['product_id']
            recs = fn(uid, max(k_values))
            if not recs: continue
            total += 1
            recs_union.update(recs)
            for k in k_values:
                if held_out in recs[:k]:
                    hits[k]     += 1
                    ndcg_sum[k] += _ndcg(recs, held_out, k)
        row_data = {}
        for k in k_values:
            row_data[f'HR@{k}']         = round(hits[k]/total, 4) if total else 0
            row_data[f'Precision@{k}']  = round(hits[k]/(total*k), 4) if total else 0
            row_data[f'NDCG@{k}']       = round(ndcg_sum[k]/total, 4) if total else 0
        row_data['Coverage@20']     = round(len(recs_union)/len(all_items), 4)
        row_data['Users_evaluated'] = total
        results[name] = row_data
        print(f'{time.time()-t0:5.1f}s')
    return pd.DataFrame(results).T

recommenders = {
    'Non-Personalized' : eval_np,
    'Content-Based'    : eval_cbf,
    'Item-Based CF'    : eval_item_cf,
    'User-Based CF'    : eval_user_cf,
    'Switching Hybrid' : eval_switching,
    'Mixed Hybrid'     : eval_mixed,
    'Weighted Hybrid'  : eval_weighted,
    'Cascade Hybrid'   : eval_cascade,
}

print(f'Running evaluation on {len(test_df)} test users...\n')
eval_results = run_evaluation(recommenders, test_df)
print('\nDone.')


Running evaluation on 1191 test users...

  Non-Personalized               2.9s
  Content-Based                 21.2s
  Item-Based CF                  8.3s
  User-Based CF                  3.5s
  Switching Hybrid               4.4s
  Mixed Hybrid                  23.8s
  Weighted Hybrid               11.9s
  Cascade Hybrid                 4.4s

Done.


In [51]:
# ── Hit Rate / Precision / NDCG / Coverage ───────────────────────────────────
metric_cols = [c for c in eval_results.columns if c != 'Users_evaluated']
print('=== Evaluation Results ===\n')
print(eval_results[metric_cols].astype(float).to_string())

# Best model per column
print('\n=== Best model per metric ===')
for col in metric_cols:
    best = eval_results[col].astype(float).idxmax()
    val  = eval_results.loc[best, col]
    print(f'  {col:<15}: {best}  ({val})')

# RMSE (rating-prediction accuracy — CF models only)
rmse_map = {
    'Item-Based CF': round(float(np.sqrt(((pd.Series(item_actuals) - pd.Series(item_preds))**2).mean())), 4),
    'User-Based CF': round(float(np.sqrt(((pd.Series(user_actuals) - pd.Series(user_preds))**2).mean())), 4),
}
print('\n=== RMSE — rating prediction (CF models only) ===')
for k, v in rmse_map.items():
    print(f'  {k:<22}: {v}')


=== Evaluation Results ===

                    HR@5  Precision@5  NDCG@5   HR@10  Precision@10  NDCG@10   HR@20  Precision@20  NDCG@20  Coverage@20
Non-Personalized  0.0000       0.0000  0.0000  0.0000        0.0000   0.0000  0.0000        0.0000   0.0000       0.0156
Content-Based     0.0487       0.0097  0.0405  0.0638        0.0064   0.0453  0.0898        0.0045   0.0516       0.8844
Item-Based CF     0.1830       0.0366  0.0869  0.1914        0.0191   0.0897  0.1973        0.0099   0.0913       0.3170
User-Based CF     0.0327       0.0065  0.0181  0.0470        0.0047   0.0226  0.0546        0.0027   0.0246       0.7704
Switching Hybrid  0.0361       0.0072  0.0215  0.0504        0.0050   0.0259  0.0579        0.0029   0.0280       0.7844
Mixed Hybrid      0.0411       0.0082  0.0174  0.2250        0.0225   0.0718  0.2250        0.0113   0.0718       0.4963
Weighted Hybrid   0.0218       0.0044  0.0147  0.0361        0.0036   0.0191  0.0521        0.0026   0.0231       0.6074
Casc

In [52]:
# ── Intra-list diversity: avg pairwise content dissimilarity within top-10 ────
def intra_list_diversity(rec_ids, k=10):
    idxs = [_product_index_map[p] for p in rec_ids[:k] if p in _product_index_map]
    if len(idxs) < 2: return 0.0
    dists = [1 - float(similarity_matrix[i, j])
             for ii, i in enumerate(idxs) for j in idxs[ii+1:]]
    return float(np.mean(dists))


sample_users = test_df.sample(200, random_state=42)
div_rows = {}
for name, fn in recommenders.items():
    divs = []
    for _, row in sample_users.iterrows():
        recs = fn(row['user_id'], 10)
        if recs: divs.append(intra_list_diversity(recs))
    div_rows[name] = round(float(np.mean(divs)), 4) if divs else 0.0

div_series = pd.Series(div_rows, name='Diversity@10')
cov_series = eval_results['Coverage@20'].astype(float).rename('Coverage@20')

summary = pd.concat([cov_series, div_series], axis=1)
print('=== Coverage & Diversity ===')
print('Coverage@20: fraction of the catalogue recommended across all test users')
print('Diversity@10: avg pairwise content dissimilarity (0=all identical, 1=all different)\n')
print(summary.to_string())


=== Coverage & Diversity ===
Coverage@20: fraction of the catalogue recommended across all test users
Diversity@10: avg pairwise content dissimilarity (0=all identical, 1=all different)

                  Coverage@20  Diversity@10
Non-Personalized       0.0156        0.7049
Content-Based          0.8844        0.1793
Item-Based CF          0.3170        0.5365
User-Based CF          0.7704        0.5599
Switching Hybrid       0.7844        0.5591
Mixed Hybrid           0.4963        0.6340
Weighted Hybrid        0.6074        0.3065
Cascade Hybrid         0.0696        0.5333


#### Results Discussion

**Key findings:**

| Model | Strength | Weakness |
|---|---|---|
| Non-Personalized | Highest coverage; safe cold-start baseline | Same list for every user; zero personalisation |
| Content-Based | Works with just 1 rating; good for niche items | Depends on feature quality; ignores collaborative signal |
| Item-Based CF | Best Hit Rate & NDCG among single models (RMSE ≈ 0.81) | Needs training ratings; fails for cold-start users |
| User-Based CF | Captures taste communities | Very sparse data → most user pairs share 0 items; poor Hit Rate |
| Switching Hybrid | Routes each user to best single model | 99.7 % of users hit the CF branch → nearly identical to User-CF |
| Mixed Hybrid | Transparent sourcing + diversity | NP items appear first → penalised at small K in offline eval |
| Weighted Hybrid | Tunable balance; integrates all signals into one ranked list | CBF column-sum scales O(catalogue × ratings); slow for large catalogues |
| Cascade Hybrid | Safe NP pool → CBF narrows by content → CF provides final re-rank | Funnel can drop relevant items if pool sizes are too small |

**Limitations:**

1. **Extreme sparsity** — Training averages ≈ 4 ratings / user after the split. User-based CF is crippled because most user pairs share no common items.
2. **Single held-out evaluation** — Holding out exactly 1 item per user means Hit Rate = Recall. Real evaluation should hold out ≥2 items or use a temporal split.
3. **Offline ≠ Online** — Offline metrics measure whether the exact held-out item was retrieved. They do not capture satisfaction, novelty, or serendipity. An A/B test is required to measure real impact.
4. **Composite user IDs** — Each `user_id` is a concatenation of multiple Amazon reviewers merged into one row. This inflates per-user rating counts and distorts user-user similarity.
5. **Content feature quality** — TF-IDF treats "USB-C cable" and "Type-C cable" as different tokens. Sentence-transformer embeddings would improve CBF quality significantly.
6. **No temporal ordering** — All ratings are treated equally regardless of age. A recency-weighted model would better reflect current preferences.